# Password Vault

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
!pip install cryptography

from cryptography.fernet import Fernet
import base64
import os
import getpass

key_file = "key.key"
vault_file = "vault.enc"

def generate_key(password):
    from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
    from cryptography.hazmat.primitives import hashes
    from cryptography.hazmat.backends import default_backend
    salt = b'static_salt'
    kdf = PBKDF2HMAC(
        algorithm=hashes.SHA256(),
        length=32,
        salt=salt,
        iterations=100000,
        backend=default_backend()
    )
    return base64.urlsafe_b64encode(kdf.derive(password.encode()))

def load_key(password):
    return generate_key(password)

def encrypt_data(data, key):
    f = Fernet(key)
    return f.encrypt(data.encode())

def decrypt_data(data, key):
    f = Fernet(key)
    return f.decrypt(data).decode()

def save_vault(data, key):
    encrypted = encrypt_data(data, key)
    with open(vault_file, 'wb') as f:
        f.write(encrypted)

def load_vault(key):
    if not os.path.exists(vault_file):
        return ""
    with open(vault_file, 'rb') as f:
        encrypted = f.read()
    return decrypt_data(encrypted, key)

def add_entry(vault, service, username, password):
    return vault + f"{service}:{username}:{password}\n"

password = input("Master password: ")
key = load_key(password)
vault = load_vault(key)

while True:
    action = input("Choose [add/view/exit]: ").lower()
    if action == "add":
        s = input("Service: ")
        u = input("Username: ")
        p = input("Password: ")
        vault = add_entry(vault, s, u, p)
        save_vault(vault, key)
    elif action == "view":
        print(vault)
    elif action == "exit":
        break
